In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import json

# Đọc dữ liệu
annonimized_path = '/content/drive/MyDrive/CS114.P11/annonimized.csv'
annonimized_df = pd.read_csv(annonimized_path)

# Chuyển đổi cột thời gian
annonimized_df['created_at'] = pd.to_datetime(annonimized_df['created_at'], format='%d/%m/%Y %H:%M')
annonimized_df['updated_at'] = pd.to_datetime(annonimized_df['updated_at'], format='%d/%m/%Y %H:%M')

# Tách thời gian và bộ nhớ từ cột judgement
def extract_time_memory(row):
    if row['status'] == 'SCORE':
        judgement = json.loads(row['judgement'])
        total_time = sum(judgement.get('times', []))
        total_memory = sum(judgement.get('mems', []))
        return total_time, total_memory
    return 0, 0

annonimized_df[['total_time', 'total_memory']] = annonimized_df.apply(
    lambda row: pd.Series(extract_time_memory(row)), axis=1
)

# Tính tổng điểm cuối cùng (is_final = 1, chọn điểm mới nhất hoặc cao nhất)
final_scores = (
    annonimized_df[annonimized_df['is_final'] == 1]
    .sort_values(by=['username', 'problem_id', 'updated_at'], ascending=[True, True, False])
    .groupby(['username', 'problem_id'])
    .agg(final_score=('pre_score', 'max'))
    .groupby('username')['final_score'].sum()
    .reset_index()
)

# Thêm thông tin về tổng điểm cuối cùng
annonimized_df = annonimized_df.merge(final_scores, on='username', how='left')

# Tính toán các đặc trưng
user_stats = annonimized_df.groupby('username').agg(
    total_pre_score=('pre_score', 'sum'),
    total_problems=('problem_id', lambda x: x[annonimized_df.loc[x.index, 'pre_score'] == 10000].nunique()),
    total_days=('created_at', lambda x: x.dt.date.nunique()),
    total_submissions=('assignment_id', 'count'),
    total_late_days=('coefficient', lambda x: (x < 100).sum()),
    avg_time=('total_time', lambda x: x[annonimized_df.loc[x.index, 'status'] == 'SCORE'].sum() / max(1, len(x))),
    avg_memory=('total_memory', lambda x: x[annonimized_df.loc[x.index, 'status'] == 'SCORE'].sum() / max(1, len(x))),
    total_assignments=('assignment_id', 'nunique'),
    total_score_submissions=('status', lambda x: (x == 'SCORE').sum()),
    total_error_submissions=('status', lambda x: (x == 'Compilation Error').sum()),
    final_total_score=('final_score', 'first')  # Tổng điểm cuối cùng từ bảng final_scores
).reset_index()

# Thay thế giá trị NaN và Inf trong các đặc trưng
user_stats.fillna(0, inplace=True)
user_stats.replace([np.inf, -np.inf], 0, inplace=True)

# Lưu kết quả ra file CSV
output_path = '/content/drive/MyDrive/CS114.P11/predict/QT/features.csv'
user_stats.to_csv(output_path, index=False)

print(f"User stats saved to {output_path}")

User stats saved to /content/drive/MyDrive/CS114.P11/predict/QT/features.csv


In [ ]:
!pip install odfpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 717.0/717.0 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for odfpy: filename=odfpy-1.4.1-py2.py3-none-any.whl size=160672 sha256=f135b4e20026a96ab27b70e351bfa9742db8273c947b803fa8c67ddb2a88bc41
  Stored in directory: /root/.cache/pip/wheels/d6/1d/c8/8c29be1d73ca42d15977c75193d9f39a98499413c2838ac54c
Successfully built odfpy


In [ ]:
import pandas as pd

# Đọc dữ liệu từ các file CSV
output_user_stats = pd.read_csv('/content/drive/MyDrive/CS114.P11/predict/QT/features.csv')
ck_public = pd.read_csv('/content/drive/MyDrive/CS114.P11/predict/QT/qt-public.csv')
# ck_public = pd.read_excel('/content/drive/MyDrive/CS114.P11/predict/TBTL/tbtl-public.ods', engine="odf")

# Kiểm tra thông tin về dữ liệu
print(output_user_stats.head())
print(ck_public.head())

                                   username  total_pre_score  total_problems  \
0  00b6dd4fc7eb817e03708c532016ef30ce564a61           809110              46   
1  00bef8afee8f3c595d535c9c03c490cac1a4f021          1421535              72   
2  01122b3ef7e59b84189e65985305f575d6bdf83c          1164882              58   
3  0134f9f410c65ad0e8c2254a7e9288670e02a183           595276              47   
4  013de369c439ab0ead8aa7da64423aa395a8be39           692766              44   

   total_days  total_submissions  total_late_days  avg_time   avg_memory  \
0          14                147                0  0.322517  2289.850340   
1          20                259                0  0.253514  2946.625483   
2          25                195                0  0.201436  2054.256410   
3          13                100                0  0.066000   374.480000   
4           8                107                3  0.743832  4987.813084   

   total_assignments  total_score_submissions  total_error_sub

In [ ]:
# Lọc những người dùng chưa có điểm TBTL trong ck-public.csv
users_with_scores = ck_public['username'].tolist()
users_no_score = output_user_stats[~output_user_stats['username'].isin(users_with_scores)]

# Lọc những người dùng có dữ liệu điểm TBTL
users_with_data = output_user_stats[output_user_stats['username'].isin(users_with_scores)]

# Kiểm tra lại dữ liệu đã lọc
print(users_no_score.head())
print(users_with_data.head())

                                     username  total_pre_score  \
3    0134f9f410c65ad0e8c2254a7e9288670e02a183           595276   
20   035f97702f2c01d26ab1fae8f39ea2f98a0caa3c           725150   
68   0aaebc88f6106684d6993c156104c1ef36cf94e0           780551   
80   0bf111a9caedf02804f6991792490e63bc21058a          1264795   
120  12887fd9a4df4ba9b88a71f3fb1d2502a75995dd          1001436   

     total_problems  total_days  total_submissions  total_late_days  avg_time  \
3                47          13                100                0  0.066000   
20               50           8                144                0  0.004236   
68               50          12                192                0  0.025104   
80               62          15                199                0  0.295980   
120              63          15                155                0  0.007871   

      avg_memory  total_assignments  total_score_submissions  \
3     374.480000                  4                 

In [ ]:
import pandas as pd
from sklearn.ensemble import VotingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Chọn các cột đặc trưng và cột mục tiêu
X = users_with_data[['total_pre_score', 'total_problems', 'total_days', 'total_submissions', 'avg_time', 'avg_memory', 'total_assignments', 'total_score_submissions', 'total_error_submissions', 'final_total_score']]
y = ck_public.set_index('username').loc[users_with_data['username'], 'TBTL']

# Chia dữ liệu thành tập huấn luyện và tập kiểm tra
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=40)

# Khởi tạo các mô hình thành phần
linear_reg = LinearRegression()
random_forest = RandomForestRegressor(random_state=42)
gradient_boost = GradientBoostingRegressor(random_state=42)

# Tạo Voting Regressor
voting_regressor = VotingRegressor(estimators=[
    ('lr', linear_reg),
    ('rf', random_forest),
    ('gb', gradient_boost)
])

# Huấn luyện mô hình Voting Regressor
voting_regressor.fit(X_train, y_train)

# Dự đoán trên tập kiểm tra
y_pred = voting_regressor.predict(X_test)

# Đánh giá mô hình
mse = mean_squared_error(y_test, y_pred)
print(f'Mean Squared Error: {mse}')

# Dự đoán cho những người dùng chưa có điểm
X_no_score = users_no_score[['total_pre_score', 'total_problems', 'total_days', 'total_submissions', 'avg_time', 'avg_memory', 'total_assignments', 'total_score_submissions', 'total_error_submissions', 'final_total_score']]
predicted_scores = voting_regressor.predict(X_no_score)

# Thêm kết quả dự đoán vào DataFrame
users_no_score['predicted_TBTL'] = predicted_scores

# Kiểm tra kết quả
print(users_no_score[['username', 'predicted_TBTL']].head())

# Xuất kết quả dự đoán vào file CSV
users_no_score[['username', 'predicted_TBTL']].to_csv('/content/drive/MyDrive/CS114.P11/predict/QT/predicted_scores111.csv', index=False)

# Định nghĩa hàm custom_round
def custom_round(value):
    decimal_part = value - int(value)  # Lấy phần thập phân
    if decimal_part < 0.25:
        return int(value)
    elif decimal_part < 0.75:
        return int(value) + 0.5
    else:
        return int(value) + 1

# Đọc lại dữ liệu dự đoán
predicted_tbtl_path = '/content/drive/MyDrive/CS114.P11/predict/QT/predicted_scores111.csv'
predicted_tbtl = pd.read_csv(predicted_tbtl_path)

# Áp dụng làm tròn
predicted_tbtl['predicted_TBTL'] = predicted_tbtl['predicted_TBTL'].apply(custom_round)

# Lưu lại file đã làm tròn
rounded_output_path = '/content/drive/MyDrive/CS114.P11/predict/QT/predicted_ck111_rounded.csv'
predicted_tbtl.to_csv(rounded_output_path, index=False)

print(f"Rounded TBTL scores saved to {rounded_output_path}")


Mean Squared Error: 2.1354076331353444
                                     username  predicted_TBTL
3    0134f9f410c65ad0e8c2254a7e9288670e02a183        8.192503
20   035f97702f2c01d26ab1fae8f39ea2f98a0caa3c        7.774845
68   0aaebc88f6106684d6993c156104c1ef36cf94e0        8.522971
80   0bf111a9caedf02804f6991792490e63bc21058a        8.521813
120  12887fd9a4df4ba9b88a71f3fb1d2502a75995dd        8.318909
Rounded TBTL scores saved to /content/drive/MyDrive/CS114.P11/predict/QT/predicted_ck111_rounded.csv


<ipython-input-37-9d22cddd0e92>:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  users_no_score['predicted_TBTL'] = predicted_scores
